In [ ]:
!pip install groq

In [ ]:

!pip install pdfplumber groq

In [3]:
#test
import pdfplumber

def extract_text_from_pdf(pdf_path):
    with pdfplumber.open(pdf_path) as pdf:
        full_text = ""
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                full_text += text + "\n"
    return full_text


test_pdf_path = r"C:\Users\yildi\Desktop\fatura şablonları\ekspofaktoring fatura şablonu.pdf"

try:
    sample_text = extract_text_from_pdf(test_pdf_path)
    print("pdf başarıyla okundu ilk bin karakter :")
    print(sample_text[:1000]) 
except Exception as e:
    print(f" Hata mesajı: {e}")

pdf başarıyla okundu ilk bin karakter :
Sayın
MÜŞTERİmiz,
Şirketimiz’in internet sitesinde yayımlanmış bulunan Faktoring Sözleşmesi ve eklerinin
aşağıda bilgilerinize sunulan genel işlem şartlarını ve Faktoring işlemlerinin
sağlıklı yürütülebilmesi amacıyla açıklayıcı hükümleri, Uluslararası Faktoring
Kurallarını (GRIF ) içerdiğini önemle bildiririz.
Faktoring işlemlerinin yurtiçi ve yurtdışı uygulamalarında, uluslararası Faktoring
kuralları (GRIF) ile Faktorlar arası anlaşmaların uygulama alanı bulacağından, uyulması
zorunlu bu kuralların ve anlaşmaların orjinal metnini de internet sitemizde bulabilir,
inceleyebilirsiniz.
Şirketimizin internet sitesinden serbestçe elde edebileceğiniz Faktoring Sözleşmesi ve
eklerini, GRIF kurallarını ve Faktorlar arası anlaşmaları, gerekli görmeniz halinde hukuk
danışmanınız ile birlikte değerlendirmenizi özelikle duyururuz.
Daha fazla bilgi almak, görüşmek istediğiniz takdirde, önceden randevu alarak şirketimizi
ziyaret etmenizden memnuniyet duyarız.

In [4]:
import os

folder_path = r"C:\Users\yildi\Desktop\fatura şablonları"

all_pdf_texts = {}

print(f"'{folder_path}' klasör taranıyor\n")

for filename in os.listdir(folder_path):
    if filename.lower().endswith('.pdf'):
        full_path = os.path.join(folder_path, filename)
        print(f"Okunan dosya: {filename}")
        
        try:
            extracted_text = extract_text_from_pdf(full_path)
            
            all_pdf_texts[filename] = extracted_text
            
        except Exception as e:
            print(f"{filename} okunurken hata oluştu. {e}")

print(f"\nİşlem Tamamlandı. Toplam {len(all_pdf_texts)} adet PDF ile sözlük oluştu.")

'C:\Users\yildi\Desktop\fatura şablonları' klasör taranıyor

Okunan dosya: alacakbildiriformu.pdf
Okunan dosya: ekspofaktoring fatura şablonu.pdf
Okunan dosya: garanti şablonu.pdf
Okunan dosya: kapital fatura şablonu.pdf
Okunan dosya: tamfinans fatura şablonu.pdf

İşlem Tamamlandı. Toplam 5 adet PDF ile sözlük oluştu.


In [ ]:
from groq import Groq
import json
import time

client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

def parse_document_with_groq(raw_text):
    
    system_prompt = (
        "Sen profesyonel bir finansal doküman analistisin. Sana gönderilen metinlerden, "
        "şablon ne olursa olsun aşağıdaki bilgileri bulup SADECE belirtilen JSON formatında çıktı vermen gerekiyor.\n"
        "Kurallar:\n"
        "1. Kelimeler farklı olabilir (Toplam, Fatura Bedeli, Matrah, Brüt vb.), bunları bizim ML veri setimizdeki isimlerle akıllıca eşleştir.\n"
        "2. Çıktıda ASLA JSON dışında açıklama veya ```json gibi markdown ibareleri olmasın. Direkt saf JSON dön.\n"
        "3. Bulamadığın veya metinde geçmeyen sözel değerler için 'UNKNOWN', sayısal değerler için '0.0' yaz.\n\n"
        "İstenen JSON Formatı (Bizim ML Sütun İsimlerimiz):\n"
        "{\n"
        "  'customer_id': 'Varsa faturadaki müşteri/borçlu numarası, yoksa UNKNOWN',\n"
        "  'invoice_amount': 'Sayısal olarak faturanın toplam tutarı/bedeli (float, örn: 154500.00)',\n"
        "  'tax_type': 'Faturadaki KDV oranı veya vergi türü (Örn: KDV18, KDV20, Muaf)',\n"
        "  'factoring_type': 'Eğer metinde geçiyorsa faktoring türü (Örn: Yurtiçi, Yurtdışı), yoksa UNKNOWN',\n"
        "  'sector': 'Faturayı kesen veya alan şirketin faaliyet gösterdiği sektör (Örn: Tekstil, Lojistik, İnşaat, Gıda)'\n"
        "}"
    )

    
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Analiz edilecek metin:\n{raw_text[:8000]}"} 
        ],
        temperature=0.0
    )
    
    try:
        
        return json.loads(response.choices[0].message.content)
    except Exception as e:
        return {"error": "JSON parse hatası", "raw_output": response.choices[0].message.content}


all_parsed_results = {}
print("Analiz başlatıldı\n")

for filename, pdf_text in all_pdf_texts.items():
    print(f"LLM İşleniyor: {filename}")
    all_parsed_results[filename] = parse_document_with_groq(pdf_text)
    time.sleep(1)

print("\n Başarılı.")

Analiz başlatıldı

LLM İşleniyor: alacakbildiriformu.pdf
LLM İşleniyor: ekspofaktoring fatura şablonu.pdf
LLM İşleniyor: garanti şablonu.pdf
LLM İşleniyor: kapital fatura şablonu.pdf
LLM İşleniyor: tamfinans fatura şablonu.pdf

 Başarılı.


In [8]:
import os
import json


output_folder = r"C:\Users\yildi\Desktop\tamfinans"
output_file_path = os.path.join(output_folder, "analiz_sonuclari.json")


with open(output_file_path, "w", encoding="utf-8") as f:
    json.dump(all_parsed_results, f, indent=4, ensure_ascii=False)

print(f"İşlem başarılı")


İşlem başarılı


In [1]:
import os
import json
import pickle

output_folder = r"C:\Users\yildi\Desktop\tamfinans"
json_file_path = os.path.join(output_folder, "analiz_sonuclari.json")
pkl_file_path = os.path.join(output_folder, "analiz_sonuclari.pkl")

with open(json_file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

with open(pkl_file_path, "wb") as f:
    pickle.dump(data, f)


print(f" {pkl_file_path}")

 C:\Users\yildi\Desktop\tamfinans\analiz_sonuclari.pkl
